In [1]:
!pip install datasets tqdm requests

# Download Hugging Face Dataset to TXT

In [ ]:
import os
from datasets import load_dataset

In [ ]:
def download_and_save_facts(dataset_name="MuskumPillerum/General-Knowledge", output_file="general_knowledge.txt", max_size_mb=1.0):
    """
    Downloads the General Knowledge dataset from Hugging Face and saves it as a plain .txt file.
    Ensures the output file size does not exceed the specified maximum size in megabytes.
    """
    print(f"Loading dataset '{dataset_name}' from Hugging Face...")
    
    # Load the dataset from the Hugging Face Hub
    try:
        dataset = load_dataset(dataset_name)
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Please ensure you have the 'datasets' library installed: pip install datasets")
        return

    # Extract the primary split automatically (typically 'train')

    split_name = list(dataset.keys())[0]
    active_dataset = dataset[split_name]
    
    print(f"Converting and saving rows to '{output_file}'...")
    max_bytes = max_size_mb * 1024 * 1024
    current_bytes = 0
    record_count = 0

    # Open the text file with UTF-8 encoding to support diverse text characters
    with open(output_file, "w", encoding="utf-8") as f:
        for row in active_dataset:
            # Extract features ('Question' and 'Answer') based on the dataset schema
            question = row.get("Question", "").strip()
            answer = row.get("Answer", "").strip()
            
            # Skip empty lines if they occur
            if not question and not answer:
                continue
                
            # Format the output structure cleanly for language model pretraining
            formatted_text = f"Question: {question}\nAnswer: {answer}\n\n"
            text_bytes = len(formatted_text.encode("utf-8"))
            
            # Enforce the strict file size barrier before writing the current record
            if current_bytes + text_bytes > max_bytes:
                print(f"Reached the maximum size limit constraint of {max_size_mb} MB.")
                break
                
            f.write(formatted_text)
            current_bytes += text_bytes
            record_count += 1

    # Print final file statistics for validation
    file_size_kb = os.path.getsize(output_file) / 1024
    print(f"Successfully saved {record_count} records to '{output_file}' ({file_size_kb:.2f} KB).")

In [6]:
# Setting max_size_mb to 0.95 gives a safe headroom buffer below 1 MB
download_and_save_facts(
    dataset_name="MuskumPillerum/General-Knowledge",
    output_file="general_knowledge_facts.txt",
    max_size_mb=0.95
)

Loading dataset 'MuskumPillerum/General-Knowledge' from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

output.json: reconstructing file:   0%|          |  0.00B / 16.2MB            

output.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/37635 [00:00<?, ? examples/s]

Converting and saving rows to 'general_knowledge_facts.txt'...
Reached the maximum size limit constraint of 0.95 MB.
Successfully saved 3475 records to 'general_knowledge_facts.txt' (972.68 KB).
